
# 16 — Multi-Indicator Shock Validation  
## Validating the Main 5-Variable POSet Beyond GDP Recovery

This notebook extends validation beyond the GDP recovery-year variable.

It runs after:

```text
15_Country_POSet_Diagnostic_Drilldown_2002_5Var.ipynb
```

## Why this notebook exists

Notebook 11 validated the POSet against GDP recovery time.  
This notebook validates the structural POSet against a broader set of post-shock macro indicators:

- GDP growth
- unemployment
- inflation
- public debt
- productivity

## Core guardrail

These are **validation outcomes**, not ordering variables.

The POSet ordering relation remains based only on the five pre-shock structural variables:

1. debt capacity
2. employment strength
3. R&D intensity
4. tertiary human capital
5. energy security

## Validation logic

For each shock-specific pre-shock baseline:

- 2007 baseline → post-shock window 2008–2012
- 2019 baseline → post-shock window 2020–2023

The notebook compares whether frontier / stronger structural profiles show more favourable post-shock macro behaviour.

## Important interpretation

This is descriptive validation, not causal inference.

Good validation evidence means:

```text
The structural POSet is directionally aligned with later macro outcomes.
```

It does **not** mean:

```text
The five ordering variables caused the recovery outcome.
```

## Main outputs

- `multi_indicator_validation_country_level.csv`
- `frontier_vs_nonfrontier_multi_indicator_validation.csv`
- `layer_multi_indicator_validation_summary.csv`
- `multi_indicator_validation_mismatch_cases.csv`
- `multi_indicator_validation_takeaways.csv`
- `multi_indicator_validation_synthesis.csv`
- `validation_evidence_matrix.csv`
- `report_ready_multi_indicator_validation_paragraphs.csv`
- `16_Multi_Indicator_Shock_Validation_Audit.xlsx`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
RECOVERY_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Multi_Indicator_Shock_Validation"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "16_Multi_Indicator_Shock_Validation"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Multi_Indicator_Shock_Validation"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Multi_Indicator_Shock_Validation"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_220702
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Multi_Indicator_Shock_Validation


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

PANEL_FILE_CANDIDATES = [
    MASTER_DIR / "master_country_year_panel.csv",
    MASTER_DIR / "master_country_year_panel_2002_2025.csv",
]

PROFILE_COUNTRY_MAP_FILE = PROFILE_DIR / "profile_country_map_with_layers.csv"
RECOVERY_VALIDATION_DATASET_FILE = RECOVERY_DIR / "recovery_validation_country_dataset.csv"

def first_existing(candidates, label):
    for path in candidates:
        if path.exists():
            print(f"Using {label}: {path}")
            return path
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n" + "\n".join(str(p) for p in candidates)
    )

PANEL_FILE = first_existing(PANEL_FILE_CANDIDATES, "master country-year panel")

if not PROFILE_COUNTRY_MAP_FILE.exists():
    raise FileNotFoundError(f"Missing profile country map: {PROFILE_COUNTRY_MAP_FILE}")

SHOCK_WINDOWS = {
    "2007": {
        "shock_label": "Global Financial Crisis",
        "baseline_year": 2007,
        "post_start": 2008,
        "post_end": 2012,
        "window_note": "Five-year post-shock window after the 2007 baseline.",
    },
    "2019": {
        "shock_label": "COVID shock",
        "baseline_year": 2019,
        "post_start": 2020,
        "post_end": 2023,
        "window_note": "Post-COVID validation window using available post-shock years.",
    },
}

VARIABLE_CANDIDATES = {
    "gdp_growth": ["gdp_growth", "GDP_growth", "gdp_growth_rate"],
    "unemployment_rate": ["unemployment_rate", "unemployment"],
    "inflation_cpi": ["inflation_cpi", "inflation_rate", "cpi_inflation"],
    "public_debt_gdp_canonical": ["public_debt_gdp_canonical", "public_debt_gdp", "debt_gdp"],
    "productivity_gdp_per_hour": ["productivity_gdp_per_hour", "productivity", "gdp_per_hour"],
}

# Validation metric definitions.
# raw_metric_value is transformed to direction_aligned_value so higher = better for cross-metric comparison.
VALIDATION_METRIC_CONFIG = [
    {
        "metric_id": "gdp_growth_post_mean",
        "source_variable_key": "gdp_growth",
        "label": "Average post-shock GDP growth",
        "calculation": "post_mean",
        "direction": "higher_is_better",
        "interpretation": "Higher average GDP growth during the post-shock window indicates stronger macro performance.",
    },
    {
        "metric_id": "unemployment_post_mean",
        "source_variable_key": "unemployment_rate",
        "label": "Average post-shock unemployment",
        "calculation": "post_mean",
        "direction": "lower_is_better",
        "interpretation": "Lower average unemployment during the post-shock window is more favourable.",
    },
    {
        "metric_id": "unemployment_change_from_baseline",
        "source_variable_key": "unemployment_rate",
        "label": "Unemployment change from baseline",
        "calculation": "post_mean_minus_baseline",
        "direction": "lower_is_better",
        "interpretation": "A smaller increase, or larger decline, in unemployment is more favourable.",
    },
    {
        "metric_id": "inflation_abs_post_mean",
        "source_variable_key": "inflation_cpi",
        "label": "Average absolute post-shock inflation",
        "calculation": "post_mean_abs",
        "direction": "lower_is_better",
        "interpretation": "Lower absolute inflation is interpreted as greater price stability.",
    },
    {
        "metric_id": "public_debt_change_from_baseline",
        "source_variable_key": "public_debt_gdp_canonical",
        "label": "Public debt change from baseline",
        "calculation": "post_mean_minus_baseline",
        "direction": "lower_is_better",
        "interpretation": "A smaller increase, or larger decline, in debt/GDP is more favourable.",
    },
    {
        "metric_id": "productivity_change_from_baseline",
        "source_variable_key": "productivity_gdp_per_hour",
        "label": "Productivity change from baseline",
        "calculation": "post_last_minus_baseline",
        "direction": "higher_is_better",
        "interpretation": "A larger productivity increase is more favourable.",
    },
]

print("Shock windows:")
for k, v in SHOCK_WINDOWS.items():
    print(k, v)

print("Validation metric count:", len(VALIDATION_METRIC_CONFIG))


Using master country-year panel: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Master_Dataset\master_country_year_panel.csv
Shock windows:
2007 {'shock_label': 'Global Financial Crisis', 'baseline_year': 2007, 'post_start': 2008, 'post_end': 2012, 'window_note': 'Five-year post-shock window after the 2007 baseline.'}
2019 {'shock_label': 'COVID shock', 'baseline_year': 2019, 'post_start': 2020, 'post_end': 2023, 'window_note': 'Post-COVID validation window using available post-shock years.'}
Validation metric count: 6


In [3]:

# ------------------------------------------------------
# HELPERS
# ------------------------------------------------------

table_inventory = []
figure_inventory = []

def clean_keys(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "country" in out.columns and "Country" not in out.columns:
        out = out.rename(columns={"country": "Country"})
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

    if "Year" not in out.columns and "year" in out.columns:
        out = out.rename(columns={"year": "Year"})

    if "Year" in out.columns:
        out["Year"] = pd.to_numeric(out["Year"], errors="coerce").astype("Int64")

    if "baseline_year" in out.columns:
        out["baseline_year"] = pd.to_numeric(out["baseline_year"], errors="coerce").astype("Int64")

    return out


def save_table(df, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    df.to_csv(processed_path, index=False)
    df.to_csv(diagnostics_path, index=False)
    df.to_csv(table_path, index=False)

    table_inventory.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(df),
        "columns": len(df.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved table:", file_name)


def save_figure(fig, file_name, title, description):
    path = FIGURES_DIR / file_name
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    figure_inventory.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "path": str(path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved figure:", file_name)


def resolve_variable(panel_columns, candidates):
    for c in candidates:
        if c in panel_columns:
            return c
    return None


def direction_align(raw_value, direction):
    if pd.isna(raw_value):
        return np.nan

    if direction == "higher_is_better":
        return raw_value

    if direction == "lower_is_better":
        return -raw_value

    raise ValueError(f"Unknown direction: {direction}")


def safe_mean(series):
    vals = pd.to_numeric(series, errors="coerce").dropna()
    return vals.mean() if len(vals) else np.nan


def safe_last_by_year(df, value_col):
    temp = df[["Year", value_col]].dropna().sort_values("Year")
    if temp.empty:
        return np.nan
    return temp[value_col].iloc[-1]


def calculate_metric(country_panel, baseline_year, post_start, post_end, source_col, calculation):
    country_panel = country_panel.copy()
    country_panel["Year"] = pd.to_numeric(country_panel["Year"], errors="coerce")

    baseline_rows = country_panel[country_panel["Year"].eq(baseline_year)]
    post_rows = country_panel[
        country_panel["Year"].between(post_start, post_end, inclusive="both")
    ].copy()

    baseline_value = safe_mean(baseline_rows[source_col]) if len(baseline_rows) else np.nan
    post_mean = safe_mean(post_rows[source_col]) if len(post_rows) else np.nan
    post_last = safe_last_by_year(post_rows, source_col) if len(post_rows) else np.nan

    if calculation == "post_mean":
        raw_metric_value = post_mean
    elif calculation == "post_mean_abs":
        raw_metric_value = safe_mean(post_rows[source_col].abs()) if len(post_rows) else np.nan
    elif calculation == "post_mean_minus_baseline":
        raw_metric_value = post_mean - baseline_value if pd.notna(post_mean) and pd.notna(baseline_value) else np.nan
    elif calculation == "post_last_minus_baseline":
        raw_metric_value = post_last - baseline_value if pd.notna(post_last) and pd.notna(baseline_value) else np.nan
    else:
        raise ValueError(f"Unknown calculation: {calculation}")

    return {
        "baseline_value": baseline_value,
        "post_mean": post_mean,
        "post_last": post_last,
        "post_observation_count": int(pd.to_numeric(post_rows[source_col], errors="coerce").notna().sum()) if len(post_rows) else 0,
        "raw_metric_value": raw_metric_value,
    }


In [4]:

# ------------------------------------------------------
# LOAD INPUT TABLES
# ------------------------------------------------------

panel = clean_keys(pd.read_csv(PANEL_FILE))
profile_map = clean_keys(pd.read_csv(PROFILE_COUNTRY_MAP_FILE))

if RECOVERY_VALIDATION_DATASET_FILE.exists():
    recovery_validation = clean_keys(pd.read_csv(RECOVERY_VALIDATION_DATASET_FILE))
else:
    recovery_validation = pd.DataFrame()

if "Year" not in panel.columns:
    raise ValueError("Master panel must contain a Year column.")

if "Country" not in panel.columns:
    raise ValueError("Master panel must contain a Country column.")

required_profile_cols = {"Country", "shock_id", "baseline_year", "layer", "is_pareto_frontier", "profile_code"}
missing_profile_cols = required_profile_cols - set(profile_map.columns)

if missing_profile_cols:
    raise ValueError(f"Profile map missing required columns: {missing_profile_cols}")

profile_map["is_pareto_frontier"] = profile_map["is_pareto_frontier"].astype(str).str.lower().isin(["true", "1", "yes"])

resolved_variables = {}

for variable_key, candidates in VARIABLE_CANDIDATES.items():
    resolved = resolve_variable(panel.columns, candidates)
    resolved_variables[variable_key] = resolved

resolved_variable_table = pd.DataFrame([
    {
        "variable_key": key,
        "resolved_column": value,
        "available": value is not None,
        "candidate_columns": ", ".join(VARIABLE_CANDIDATES[key]),
    }
    for key, value in resolved_variables.items()
])

save_table(
    resolved_variable_table,
    "multi_indicator_resolved_variables.csv",
    "Resolved validation variables",
    "Maps validation variable keys to columns available in the master country-year panel.",
)

print("Panel shape:", panel.shape)
print("Profile map shape:", profile_map.shape)
display(resolved_variable_table)
display(profile_map.head())


Saved table: multi_indicator_resolved_variables.csv
Panel shape: (3499, 26)
Profile map shape: (60, 19)


,variable_key,resolved_column,available,candidate_columns
0,gdp_growth,gdp_growth,True,"gdp_growth, GDP_growth, gdp_growth_rate"
1,unemployment_rate,unemployment_rate,True,"unemployment_rate, unemployment"
2,inflation_cpi,inflation_cpi,True,"inflation_cpi, inflation_rate, cpi_inflation"
3,public_debt_gdp_canonical,public_debt_gdp_canonical,True,"public_debt_gdp_canonical, public_debt_gdp, de..."
4,productivity_gdp_per_hour,productivity_gdp_per_hour,True,"productivity_gdp_per_hour, productivity, gdp_p..."


,Country,shock_id,baseline_year,profile_code,profile_digit_sum,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,main_variable_set,levels,layer,is_pareto_frontier
0,AUT,2007,2007,2-4-5-3-2,16.0000,2,4,5,3,2,38.5303,80.6046,68.7303,33.2757,22.6201,baseline_5_variables,5,2,False
1,BEL,2007,2007,1-2-4-4-1,12.0000,1,2,4,4,1,16.9811,46.5777,48.5360,53.5113,9.6139,baseline_5_variables,5,3,False
2,CAN,2007,2007,3-3-4-5-5,20.0000,3,3,4,5,5,64.9865,63.6697,50.3901,100.0000,100.0000,baseline_5_variables,5,1,True
3,CZE,2007,2007,4-3-3-1-5,16.0000,4,3,3,1,5,76.7627,74.6107,29.3904,0.4466,50.5562,baseline_5_variables,5,2,False
4,DEU,2007,2007,2-1-4-2-3,12.0000,2,1,4,2,3,40.6157,31.1478,68.2315,30.9850,28.5833,baseline_5_variables,5,4,False


In [5]:

# ------------------------------------------------------
# BUILD COUNTRY-LEVEL MULTI-INDICATOR VALIDATION DATASET
# ------------------------------------------------------

country_rows = []

for _, profile_row in profile_map.iterrows():
    country = profile_row["Country"]
    shock_id = str(profile_row["shock_id"])

    if shock_id not in SHOCK_WINDOWS:
        continue

    shock_cfg = SHOCK_WINDOWS[shock_id]
    baseline_year = int(shock_cfg["baseline_year"])
    post_start = int(shock_cfg["post_start"])
    post_end = int(shock_cfg["post_end"])

    country_panel = panel[panel["Country"].eq(country)].copy()

    for metric in VALIDATION_METRIC_CONFIG:
        source_key = metric["source_variable_key"]
        source_col = resolved_variables.get(source_key)

        if source_col is None:
            country_rows.append({
                "Country": country,
                "shock_id": shock_id,
                "shock_label": shock_cfg["shock_label"],
                "baseline_year": baseline_year,
                "post_start": post_start,
                "post_end": post_end,
                "profile_code": profile_row.get("profile_code"),
                "layer": profile_row.get("layer"),
                "is_pareto_frontier": profile_row.get("is_pareto_frontier"),
                "metric_id": metric["metric_id"],
                "metric_label": metric["label"],
                "source_variable_key": source_key,
                "source_column": np.nan,
                "direction": metric["direction"],
                "calculation": metric["calculation"],
                "baseline_value": np.nan,
                "post_mean": np.nan,
                "post_last": np.nan,
                "post_observation_count": 0,
                "raw_metric_value": np.nan,
                "direction_aligned_value": np.nan,
                "interpretation": metric["interpretation"],
                "missing_reason": "source_column_not_available",
            })
            continue

        if source_col not in country_panel.columns:
            missing_reason = "source_column_missing_for_country_panel"
            calc_result = {
                "baseline_value": np.nan,
                "post_mean": np.nan,
                "post_last": np.nan,
                "post_observation_count": 0,
                "raw_metric_value": np.nan,
            }
        else:
            calc_result = calculate_metric(
                country_panel=country_panel,
                baseline_year=baseline_year,
                post_start=post_start,
                post_end=post_end,
                source_col=source_col,
                calculation=metric["calculation"],
            )
            missing_reason = "" if pd.notna(calc_result["raw_metric_value"]) else "insufficient_metric_data"

        raw_metric_value = calc_result["raw_metric_value"]
        direction_aligned_value = direction_align(raw_metric_value, metric["direction"])

        country_rows.append({
            "Country": country,
            "shock_id": shock_id,
            "shock_label": shock_cfg["shock_label"],
            "baseline_year": baseline_year,
            "post_start": post_start,
            "post_end": post_end,
            "profile_code": profile_row.get("profile_code"),
            "layer": profile_row.get("layer"),
            "is_pareto_frontier": profile_row.get("is_pareto_frontier"),
            "metric_id": metric["metric_id"],
            "metric_label": metric["label"],
            "source_variable_key": source_key,
            "source_column": source_col,
            "direction": metric["direction"],
            "calculation": metric["calculation"],
            "baseline_value": calc_result["baseline_value"],
            "post_mean": calc_result["post_mean"],
            "post_last": calc_result["post_last"],
            "post_observation_count": calc_result["post_observation_count"],
            "raw_metric_value": raw_metric_value,
            "direction_aligned_value": direction_aligned_value,
            "interpretation": metric["interpretation"],
            "missing_reason": missing_reason,
        })

multi_indicator_validation_country_level = pd.DataFrame(country_rows)

multi_indicator_validation_country_level["layer"] = pd.to_numeric(
    multi_indicator_validation_country_level["layer"],
    errors="coerce",
).astype("Int64")

save_table(
    multi_indicator_validation_country_level,
    "multi_indicator_validation_country_level.csv",
    "Multi-indicator validation country level",
    "Country-metric-level validation dataset for post-shock macro indicators.",
)

display(multi_indicator_validation_country_level.head(30))


Saved table: multi_indicator_validation_country_level.csv


,Country,shock_id,shock_label,baseline_year,post_start,post_end,profile_code,layer,is_pareto_frontier,metric_id,metric_label,source_variable_key,source_column,direction,calculation,baseline_value,post_mean,post_last,post_observation_count,raw_metric_value,direction_aligned_value,interpretation,missing_reason
0,AUT,2007,Global Financial Crisis,2007,2008,2012,2-4-5-3-2,2,False,gdp_growth_post_mean,Average post-shock GDP growth,gdp_growth,gdp_growth,higher_is_better,post_mean,3.7752,0.6463,0.6282,5,0.6463,0.6463,Higher average GDP growth during the post-shoc...,
1,AUT,2007,Global Financial Crisis,2007,2008,2012,2-4-5-3-2,2,False,unemployment_post_mean,Average post-shock unemployment,unemployment_rate,unemployment_rate,lower_is_better,post_mean,4.8580,4.7338,4.8630,5,4.7338,-4.7338,Lower average unemployment during the post-sho...,
2,AUT,2007,Global Financial Crisis,2007,2008,2012,2-4-5-3-2,2,False,unemployment_change_from_baseline,Unemployment change from baseline,unemployment_rate,unemployment_rate,lower_is_better,post_mean_minus_baseline,4.8580,4.7338,4.8630,5,-0.1242,0.1242,"A smaller increase, or larger decline, in unem...",
3,AUT,2007,Global Financial Crisis,2007,2008,2012,2-4-5-3-2,2,False,inflation_abs_post_mean,Average absolute post-shock inflation,inflation_cpi,inflation_cpi,lower_is_better,post_mean_abs,2.1686,2.2616,2.4857,5,2.2616,-2.2616,Lower absolute inflation is interpreted as gre...,
4,AUT,2007,Global Financial Crisis,2007,2008,2012,2-4-5-3-2,2,False,public_debt_change_from_baseline,Public debt change from baseline,public_debt_gdp_canonical,public_debt_gdp_canonical,lower_is_better,post_mean_minus_baseline,65.8000,80.0600,82.9000,5,14.2600,-14.2600,"A smaller increase, or larger decline, in debt...",
5,AUT,2007,Global Financial Crisis,2007,2008,2012,2-4-5-3-2,2,False,productivity_change_from_baseline,Productivity change from baseline,productivity_gdp_per_hour,productivity_gdp_per_hour,higher_is_better,post_last_minus_baseline,76.8401,77.4446,78.5899,5,1.7499,1.7499,A larger productivity increase is more favoura...,
6,BEL,2007,Global Financial Crisis,2007,2008,2012,1-2-4-4-1,3,False,gdp_growth_post_mean,Average post-shock GDP growth,gdp_growth,gdp_growth,higher_is_better,post_mean,3.6769,0.6795,0.2158,5,0.6795,0.6795,Higher average GDP growth during the post-shoc...,
7,BEL,2007,Global Financial Crisis,2007,2008,2012,1-2-4-4-1,3,False,unemployment_post_mean,Average post-shock unemployment,unemployment_rate,unemployment_rate,lower_is_better,post_mean,7.4580,7.5700,7.5400,5,7.5700,-7.5700,Lower average unemployment during the post-sho...,
8,BEL,2007,Global Financial Crisis,2007,2008,2012,1-2-4-4-1,3,False,unemployment_change_from_baseline,Unemployment change from baseline,unemployment_rate,unemployment_rate,lower_is_better,post_mean_minus_baseline,7.4580,7.5700,7.5400,5,0.1120,-0.1120,"A smaller increase, or larger decline, in unem...",
9,BEL,2007,Global Financial Crisis,2007,2008,2012,1-2-4-4-1,3,False,inflation_abs_post_mean,Average absolute post-shock inflation,inflation_cpi,inflation_cpi,lower_is_better,post_mean_abs,1.8231,2.5995,2.8397,5,2.6207,-2.6207,Lower absolute inflation is interpreted as gre...,


In [6]:

# ------------------------------------------------------
# FRONTIER VS NON-FRONTIER VALIDATION
# ------------------------------------------------------

valid_country_metrics = multi_indicator_validation_country_level[
    multi_indicator_validation_country_level["raw_metric_value"].notna()
].copy()

frontier_summary = (
    valid_country_metrics
    .groupby(["shock_id", "shock_label", "metric_id", "metric_label", "direction", "is_pareto_frontier"])
    .agg(
        countries=("Country", "nunique"),
        mean_raw_metric_value=("raw_metric_value", "mean"),
        median_raw_metric_value=("raw_metric_value", "median"),
        mean_direction_aligned_value=("direction_aligned_value", "mean"),
        median_direction_aligned_value=("direction_aligned_value", "median"),
        country_list=("Country", lambda x: ", ".join(sorted(x))),
    )
    .reset_index()
)

frontier_vs_rows = []

for keys, group in frontier_summary.groupby(["shock_id", "shock_label", "metric_id", "metric_label", "direction"]):
    shock_id, shock_label, metric_id, metric_label, direction = keys

    frontier = group[group["is_pareto_frontier"].eq(True)]
    nonfrontier = group[group["is_pareto_frontier"].eq(False)]

    if len(frontier) and len(nonfrontier):
        frontier_mean = frontier["mean_raw_metric_value"].iloc[0]
        nonfrontier_mean = nonfrontier["mean_raw_metric_value"].iloc[0]

        frontier_aligned_mean = frontier["mean_direction_aligned_value"].iloc[0]
        nonfrontier_aligned_mean = nonfrontier["mean_direction_aligned_value"].iloc[0]

        frontier_advantage_direction_aligned = frontier_aligned_mean - nonfrontier_aligned_mean
        evidence_direction = "frontier_better" if frontier_advantage_direction_aligned > 0 else (
            "nonfrontier_better" if frontier_advantage_direction_aligned < 0 else "similar"
        )

        frontier_vs_rows.append({
            "shock_id": shock_id,
            "shock_label": shock_label,
            "metric_id": metric_id,
            "metric_label": metric_label,
            "direction": direction,
            "frontier_countries": int(frontier["countries"].iloc[0]),
            "nonfrontier_countries": int(nonfrontier["countries"].iloc[0]),
            "frontier_mean_raw": frontier_mean,
            "nonfrontier_mean_raw": nonfrontier_mean,
            "frontier_minus_nonfrontier_raw": frontier_mean - nonfrontier_mean,
            "frontier_mean_direction_aligned": frontier_aligned_mean,
            "nonfrontier_mean_direction_aligned": nonfrontier_aligned_mean,
            "frontier_advantage_direction_aligned": frontier_advantage_direction_aligned,
            "evidence_direction": evidence_direction,
        })

frontier_vs_nonfrontier_multi_indicator_validation = pd.DataFrame(frontier_vs_rows)

save_table(
    frontier_vs_nonfrontier_multi_indicator_validation,
    "frontier_vs_nonfrontier_multi_indicator_validation.csv",
    "Frontier vs non-frontier multi-indicator validation",
    "Compares post-shock macro outcomes between POSet frontier and non-frontier countries.",
)

display(frontier_vs_nonfrontier_multi_indicator_validation)


Saved table: frontier_vs_nonfrontier_multi_indicator_validation.csv


,shock_id,shock_label,metric_id,metric_label,direction,frontier_countries,nonfrontier_countries,frontier_mean_raw,nonfrontier_mean_raw,frontier_minus_nonfrontier_raw,frontier_mean_direction_aligned,nonfrontier_mean_direction_aligned,frontier_advantage_direction_aligned,evidence_direction
0,2007,Global Financial Crisis,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,8,17,-0.1021,-0.3229,0.2207,-0.1021,-0.3229,0.2207,frontier_better
1,2007,Global Financial Crisis,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,8,17,2.6283,2.8827,-0.2544,-2.6283,-2.8827,0.2544,frontier_better
2,2007,Global Financial Crisis,productivity_change_from_baseline,Productivity change from baseline,higher_is_better,8,17,0.3383,2.1081,-1.7698,0.3383,2.1081,-1.7698,nonfrontier_better
3,2007,Global Financial Crisis,public_debt_change_from_baseline,Public debt change from baseline,lower_is_better,8,17,16.5081,19.2212,-2.7131,-16.5081,-19.2212,2.7131,frontier_better
4,2007,Global Financial Crisis,unemployment_change_from_baseline,Unemployment change from baseline,lower_is_better,8,17,2.6391,3.2240,-0.5850,-2.6391,-3.2240,0.5850,frontier_better
5,2007,Global Financial Crisis,unemployment_post_mean,Average post-shock unemployment,lower_is_better,8,17,7.5892,10.1860,-2.5968,-7.5892,-10.1860,2.5968,frontier_better
6,2019,COVID shock,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,13,19,1.9782,1.7401,0.2381,1.9782,1.7401,0.2381,frontier_better
7,2019,COVID shock,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,13,21,4.7430,7.0190,-2.2760,-4.7430,-7.0190,2.2760,frontier_better
8,2019,COVID shock,productivity_change_from_baseline,Productivity change from baseline,higher_is_better,13,21,2.8299,2.0915,0.7385,2.8299,2.0915,0.7385,frontier_better
9,2019,COVID shock,public_debt_change_from_baseline,Public debt change from baseline,lower_is_better,13,22,5.8251,7.9644,-2.1393,-5.8251,-7.9644,2.1393,frontier_better


In [7]:

# ------------------------------------------------------
# LAYER-LEVEL MULTI-INDICATOR VALIDATION
# ------------------------------------------------------

layer_multi_indicator_validation_summary = (
    valid_country_metrics
    .groupby(["shock_id", "shock_label", "layer", "metric_id", "metric_label", "direction"])
    .agg(
        countries=("Country", "nunique"),
        mean_raw_metric_value=("raw_metric_value", "mean"),
        median_raw_metric_value=("raw_metric_value", "median"),
        mean_direction_aligned_value=("direction_aligned_value", "mean"),
        median_direction_aligned_value=("direction_aligned_value", "median"),
        country_list=("Country", lambda x: ", ".join(sorted(x))),
    )
    .reset_index()
    .sort_values(["shock_id", "metric_id", "layer"])
)

layer_trend_rows = []

for keys, group in valid_country_metrics.groupby(["shock_id", "metric_id", "metric_label", "direction"]):
    shock_id, metric_id, metric_label, direction = keys

    valid = group[["layer", "direction_aligned_value"]].dropna().copy()

    if len(valid) >= 3 and valid["layer"].nunique() > 1 and valid["direction_aligned_value"].nunique() > 1:
        corr = valid["layer"].corr(valid["direction_aligned_value"], method="spearman")
    else:
        corr = np.nan

    layer_trend_rows.append({
        "shock_id": shock_id,
        "metric_id": metric_id,
        "metric_label": metric_label,
        "direction": direction,
        "layer_direction_aligned_spearman": corr,
        "interpretation": (
            "Negative means deeper/worse POSet layers tend to have worse direction-aligned validation outcomes."
            if pd.notna(corr) else "Insufficient variation for layer trend."
        ),
    })

layer_multi_indicator_trend_summary = pd.DataFrame(layer_trend_rows)

save_table(
    layer_multi_indicator_validation_summary,
    "layer_multi_indicator_validation_summary.csv",
    "Layer multi-indicator validation summary",
    "Post-shock macro validation outcomes summarized by POSet layer.",
)

save_table(
    layer_multi_indicator_trend_summary,
    "layer_multi_indicator_trend_summary.csv",
    "Layer multi-indicator trend summary",
    "Spearman-style layer trend for direction-aligned validation outcomes.",
)

display(layer_multi_indicator_validation_summary.head(40))
display(layer_multi_indicator_trend_summary)


Saved table: layer_multi_indicator_validation_summary.csv
Saved table: layer_multi_indicator_trend_summary.csv


,shock_id,shock_label,layer,metric_id,metric_label,direction,countries,mean_raw_metric_value,median_raw_metric_value,mean_direction_aligned_value,median_direction_aligned_value,country_list
0,2007,Global Financial Crisis,1,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,8,-0.1021,-0.2500,-0.1021,-0.2500,"CAN, DNK, EST, FIN, GBR, LUX, SVN, USA"
6,2007,Global Financial Crisis,2,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,7,-0.1589,0.1133,-0.1589,0.1133,"AUT, CZE, ESP, IRL, LTU, NLD, SWE"
12,2007,Global Financial Crisis,3,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,5,0.1013,0.4355,0.1013,0.4355,"BEL, FRA, ITA, LVA, POL"
18,2007,Global Financial Crisis,4,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,3,0.6919,0.7399,0.6919,0.7399,"DEU, HUN, SVK"
24,2007,Global Financial Crisis,5,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,2,-3.4793,-3.4793,-3.4793,-3.4793,"GRC, PRT"
1,2007,Global Financial Crisis,1,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,8,2.6283,2.4306,-2.6283,-2.4306,"CAN, DNK, EST, FIN, GBR, LUX, SVN, USA"
7,2007,Global Financial Crisis,2,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,7,2.6691,2.3611,-2.6691,-2.3611,"AUT, CZE, ESP, IRL, LTU, NLD, SWE"
13,2007,Global Financial Crisis,3,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,5,3.1175,2.6207,-3.1175,-2.6207,"BEL, FRA, ITA, LVA, POL"
19,2007,Global Financial Crisis,4,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,3,3.1677,2.9391,-3.1677,-2.9391,"DEU, HUN, SVK"
25,2007,Global Financial Crisis,5,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,2,2.6160,2.6160,-2.6160,-2.6160,"GRC, PRT"


,shock_id,metric_id,metric_label,direction,layer_direction_aligned_spearman,interpretation
0,2007,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,-0.1264,Negative means deeper/worse POSet layers tend ...
1,2007,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,-0.1188,Negative means deeper/worse POSet layers tend ...
2,2007,productivity_change_from_baseline,Productivity change from baseline,higher_is_better,0.0803,Negative means deeper/worse POSet layers tend ...
3,2007,public_debt_change_from_baseline,Public debt change from baseline,lower_is_better,-0.0894,Negative means deeper/worse POSet layers tend ...
4,2007,unemployment_change_from_baseline,Unemployment change from baseline,lower_is_better,0.0354,Negative means deeper/worse POSet layers tend ...
5,2007,unemployment_post_mean,Average post-shock unemployment,lower_is_better,-0.4098,Negative means deeper/worse POSet layers tend ...
6,2019,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,-0.0378,Negative means deeper/worse POSet layers tend ...
7,2019,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,-0.1752,Negative means deeper/worse POSet layers tend ...
8,2019,productivity_change_from_baseline,Productivity change from baseline,higher_is_better,0.1455,Negative means deeper/worse POSet layers tend ...
9,2019,public_debt_change_from_baseline,Public debt change from baseline,lower_is_better,-0.0900,Negative means deeper/worse POSet layers tend ...


In [8]:

# ------------------------------------------------------
# VALIDATION TAKEAWAYS AND EVIDENCE MATRIX
# ------------------------------------------------------

takeaway_rows = []

for shock_id, group in frontier_vs_nonfrontier_multi_indicator_validation.groupby("shock_id"):
    available_metrics = len(group)
    frontier_better = int((group["evidence_direction"] == "frontier_better").sum())
    nonfrontier_better = int((group["evidence_direction"] == "nonfrontier_better").sum())
    similar = int((group["evidence_direction"] == "similar").sum())

    takeaway_rows.append({
        "shock_id": shock_id,
        "available_metrics": available_metrics,
        "frontier_better_metrics": frontier_better,
        "nonfrontier_better_metrics": nonfrontier_better,
        "similar_metrics": similar,
        "frontier_better_share": frontier_better / available_metrics if available_metrics else np.nan,
        "headline": (
            "Frontier countries outperform non-frontier countries on most available validation metrics."
            if frontier_better > nonfrontier_better else
            "Validation evidence is mixed across available macro indicators."
        ),
        "interpretation_caution": "Descriptive validation only; no causal claim.",
    })

multi_indicator_validation_takeaways = pd.DataFrame(takeaway_rows)

validation_evidence_matrix = frontier_vs_nonfrontier_multi_indicator_validation[
    [
        "shock_id", "metric_id", "metric_label", "direction",
        "frontier_advantage_direction_aligned", "evidence_direction",
    ]
].copy()

validation_evidence_matrix["evidence_label"] = np.select(
    [
        validation_evidence_matrix["evidence_direction"].eq("frontier_better"),
        validation_evidence_matrix["evidence_direction"].eq("nonfrontier_better"),
        validation_evidence_matrix["evidence_direction"].eq("similar"),
    ],
    [
        "supports_structural_frontier",
        "contradicts_structural_frontier",
        "neutral_or_similar",
    ],
    default="unknown",
)

multi_indicator_validation_synthesis = pd.DataFrame([
    {
        "finding_area": "Multi-indicator validation",
        "finding": "Validation extends beyond GDP recovery by checking GDP growth, unemployment, inflation, debt, and productivity outcomes.",
        "report_use": "Validation methodology",
    },
    {
        "finding_area": "Outcome leakage control",
        "finding": "All validation indicators are evaluated after the POSet is constructed and are not used in the ordering relation.",
        "report_use": "Methodology safeguard",
    },
    {
        "finding_area": "Frontier evidence",
        "finding": "The evidence matrix indicates how often POSet frontier countries outperform non-frontier countries across macro outcomes.",
        "report_use": "Results and discussion",
    },
    {
        "finding_area": "Interpretation",
        "finding": "Mixed evidence or mismatch cases should be interpreted as evidence of multidimensional complexity, not as failure of the method.",
        "report_use": "Discussion and limitations",
    },
])

save_table(
    multi_indicator_validation_takeaways,
    "multi_indicator_validation_takeaways.csv",
    "Multi-indicator validation takeaways",
    "Shock-level validation takeaways across all available macro indicators.",
)

save_table(
    validation_evidence_matrix,
    "validation_evidence_matrix.csv",
    "Validation evidence matrix",
    "Metric-level evidence matrix for frontier vs non-frontier validation.",
)

save_table(
    multi_indicator_validation_synthesis,
    "multi_indicator_validation_synthesis.csv",
    "Multi-indicator validation synthesis",
    "High-level synthesis of multi-indicator validation results.",
)

display(multi_indicator_validation_takeaways)
display(validation_evidence_matrix)


Saved table: multi_indicator_validation_takeaways.csv
Saved table: validation_evidence_matrix.csv
Saved table: multi_indicator_validation_synthesis.csv


,shock_id,available_metrics,frontier_better_metrics,nonfrontier_better_metrics,similar_metrics,frontier_better_share,headline,interpretation_caution
0,2007,6,5,1,0,0.8333,Frontier countries outperform non-frontier cou...,Descriptive validation only; no causal claim.
1,2019,6,5,1,0,0.8333,Frontier countries outperform non-frontier cou...,Descriptive validation only; no causal claim.


,shock_id,metric_id,metric_label,direction,frontier_advantage_direction_aligned,evidence_direction,evidence_label
0,2007,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,0.2207,frontier_better,supports_structural_frontier
1,2007,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,0.2544,frontier_better,supports_structural_frontier
2,2007,productivity_change_from_baseline,Productivity change from baseline,higher_is_better,-1.7698,nonfrontier_better,contradicts_structural_frontier
3,2007,public_debt_change_from_baseline,Public debt change from baseline,lower_is_better,2.7131,frontier_better,supports_structural_frontier
4,2007,unemployment_change_from_baseline,Unemployment change from baseline,lower_is_better,0.5850,frontier_better,supports_structural_frontier
5,2007,unemployment_post_mean,Average post-shock unemployment,lower_is_better,2.5968,frontier_better,supports_structural_frontier
6,2019,gdp_growth_post_mean,Average post-shock GDP growth,higher_is_better,0.2381,frontier_better,supports_structural_frontier
7,2019,inflation_abs_post_mean,Average absolute post-shock inflation,lower_is_better,2.2760,frontier_better,supports_structural_frontier
8,2019,productivity_change_from_baseline,Productivity change from baseline,higher_is_better,0.7385,frontier_better,supports_structural_frontier
9,2019,public_debt_change_from_baseline,Public debt change from baseline,lower_is_better,2.1393,frontier_better,supports_structural_frontier


In [9]:

# ------------------------------------------------------
# MISMATCH CASES
# ------------------------------------------------------
# Frontier-but-weak: frontier country below median direction-aligned outcome.
# Deep-layer-but-strong: deepest-layer country above median direction-aligned outcome.

mismatch_rows = []

for keys, group in valid_country_metrics.groupby(["shock_id", "metric_id", "metric_label"]):
    shock_id, metric_id, metric_label = keys

    metric_median = group["direction_aligned_value"].median()
    max_layer = group["layer"].max()

    frontier_weak = group[
        group["is_pareto_frontier"].eq(True)
        & group["direction_aligned_value"].notna()
        & (group["direction_aligned_value"] < metric_median)
    ].copy()

    for _, row in frontier_weak.iterrows():
        mismatch_rows.append({
            "Country": row["Country"],
            "shock_id": shock_id,
            "metric_id": metric_id,
            "metric_label": metric_label,
            "case_type": "frontier_but_weaker_validation_outcome",
            "layer": row["layer"],
            "profile_code": row["profile_code"],
            "raw_metric_value": row["raw_metric_value"],
            "direction_aligned_value": row["direction_aligned_value"],
            "metric_median_direction_aligned": metric_median,
            "interpretation": "Structurally frontier profile but weaker than median validation outcome on this metric.",
        })

    deep_strong = group[
        group["layer"].eq(max_layer)
        & group["direction_aligned_value"].notna()
        & (group["direction_aligned_value"] > metric_median)
    ].copy()

    for _, row in deep_strong.iterrows():
        mismatch_rows.append({
            "Country": row["Country"],
            "shock_id": shock_id,
            "metric_id": metric_id,
            "metric_label": metric_label,
            "case_type": "deep_layer_but_stronger_validation_outcome",
            "layer": row["layer"],
            "profile_code": row["profile_code"],
            "raw_metric_value": row["raw_metric_value"],
            "direction_aligned_value": row["direction_aligned_value"],
            "metric_median_direction_aligned": metric_median,
            "interpretation": "Structurally deeper profile but stronger than median validation outcome on this metric.",
        })

multi_indicator_validation_mismatch_cases = pd.DataFrame(mismatch_rows)

if multi_indicator_validation_mismatch_cases.empty:
    multi_indicator_validation_mismatch_cases = pd.DataFrame(columns=[
        "Country", "shock_id", "metric_id", "metric_label", "case_type",
        "layer", "profile_code", "raw_metric_value", "direction_aligned_value",
        "metric_median_direction_aligned", "interpretation",
    ])

save_table(
    multi_indicator_validation_mismatch_cases,
    "multi_indicator_validation_mismatch_cases.csv",
    "Multi-indicator validation mismatch cases",
    "Country-metric cases where structural POSet position and validation outcome diverge.",
)

display(multi_indicator_validation_mismatch_cases.head(50))


Saved table: multi_indicator_validation_mismatch_cases.csv


,Country,shock_id,metric_id,metric_label,case_type,layer,profile_code,raw_metric_value,direction_aligned_value,metric_median_direction_aligned,interpretation
0,DNK,2007,gdp_growth_post_mean,Average post-shock GDP growth,frontier_but_weaker_validation_outcome,1,4-5-5-4-5,-0.5007,-0.5007,0.0007,Structurally frontier profile but weaker than ...
1,EST,2007,gdp_growth_post_mean,Average post-shock GDP growth,frontier_but_weaker_validation_outcome,1,5-5-2-5-5,-1.2054,-1.2054,0.0007,Structurally frontier profile but weaker than ...
2,FIN,2007,gdp_growth_post_mean,Average post-shock GDP growth,frontier_but_weaker_validation_outcome,1,3-2-5-5-3,-0.6512,-0.6512,0.0007,Structurally frontier profile but weaker than ...
3,SVN,2007,gdp_growth_post_mean,Average post-shock GDP growth,frontier_but_weaker_validation_outcome,1,5-4-3-2-4,-1.0733,-1.0733,0.0007,Structurally frontier profile but weaker than ...
4,EST,2007,inflation_abs_post_mean,Average absolute post-shock inflation,frontier_but_weaker_validation_outcome,1,5-5-2-5-5,4.4656,-4.4656,-2.4377,Structurally frontier profile but weaker than ...
5,GBR,2007,inflation_abs_post_mean,Average absolute post-shock inflation,frontier_but_weaker_validation_outcome,1,1-4-3-5-5,2.8800,-2.8800,-2.4377,Structurally frontier profile but weaker than ...
6,SVN,2007,inflation_abs_post_mean,Average absolute post-shock inflation,frontier_but_weaker_validation_outcome,1,5-4-3-2-4,2.5376,-2.5376,-2.4377,Structurally frontier profile but weaker than ...
7,PRT,2007,inflation_abs_post_mean,Average absolute post-shock inflation,deep_layer_but_stronger_validation_outcome,5,1-2-2-1-1,2.2506,-2.2506,-2.4377,Structurally deeper profile but stronger than ...
8,FIN,2007,productivity_change_from_baseline,Productivity change from baseline,frontier_but_weaker_validation_outcome,1,3-2-5-5-3,-1.6020,-1.6020,1.2585,Structurally frontier profile but weaker than ...
9,GBR,2007,productivity_change_from_baseline,Productivity change from baseline,frontier_but_weaker_validation_outcome,1,1-4-3-5-5,-0.1366,-0.1366,1.2585,Structurally frontier profile but weaker than ...


In [10]:

# ------------------------------------------------------
# REPORT-READY PARAGRAPHS
# ------------------------------------------------------

paragraph_rows = []

for _, row in multi_indicator_validation_takeaways.iterrows():
    paragraph_rows.append({
        "topic": f"Multi-indicator validation {row['shock_id']}",
        "report_text": (
            f"For the {row['shock_id']} shock, multi-indicator validation was computed across "
            f"{int(row['available_metrics'])} available macro outcome metrics. Frontier countries show more favourable "
            f"outcomes than non-frontier countries on {int(row['frontier_better_metrics'])} metrics, while "
            f"non-frontier countries perform better on {int(row['nonfrontier_better_metrics'])} metrics. "
            f"This evidence is descriptive and does not imply causality."
        ),
    })

paragraph_rows.extend([
    {
        "topic": "Validation design",
        "report_text": (
            "The multi-indicator validation step evaluates post-shock macro outcomes after the POSet has already "
            "been constructed from pre-shock structural variables. This prevents leakage from validation outcomes "
            "into the ordering relation."
        ),
    },
    {
        "topic": "Validation variables",
        "report_text": (
            "The validation indicators include GDP growth, unemployment, inflation stability, public debt, and "
            "productivity. They are selected because they describe different dimensions of post-shock macroeconomic "
            "performance rather than the same structural capacities used for ordering."
        ),
    },
    {
        "topic": "Interpretation of mismatch cases",
        "report_text": (
            "Mismatch cases are expected in a multidimensional resilience framework. A frontier country may underperform "
            "on a particular outcome, or a deeper-layer country may perform strongly, because recovery also depends on "
            "shock exposure, policy response, sectoral composition, and factors outside the selected structural variables."
        ),
    },
])

report_ready_multi_indicator_validation_paragraphs = pd.DataFrame(paragraph_rows)

methodological_decision_updates_multi_indicator_validation = pd.DataFrame([
    {
        "decision": "Use multi-indicator validation after POSet construction",
        "choice": "GDP growth, unemployment, inflation, public debt, productivity",
        "reason": "Checks whether structural POSet position aligns with broader post-shock macro outcomes.",
        "risk_controlled": "Avoids relying only on GDP recovery years.",
    },
    {
        "decision": "Keep validation indicators outside ordering",
        "choice": "validation only",
        "reason": "They are post-shock outcomes or macro outcome metrics.",
        "risk_controlled": "Prevents outcome leakage into the POSet.",
    },
    {
        "decision": "Use shock-specific post windows",
        "choice": "2008–2012 for 2007 shock; 2020–2023 for 2019 shock",
        "reason": "Captures near/post-shock macro adjustment windows while preserving pre-shock baseline ordering.",
        "risk_controlled": "Separates structural baseline from validation horizon.",
    },
])

save_table(
    report_ready_multi_indicator_validation_paragraphs,
    "report_ready_multi_indicator_validation_paragraphs.csv",
    "Report-ready multi-indicator validation paragraphs",
    "Report-ready explanatory text for multi-indicator validation.",
)

save_table(
    methodological_decision_updates_multi_indicator_validation,
    "methodological_decision_updates_multi_indicator_validation.csv",
    "Methodological decision updates multi-indicator validation",
    "Decision-log entries for multi-indicator validation design.",
)

display(report_ready_multi_indicator_validation_paragraphs)
display(methodological_decision_updates_multi_indicator_validation)


Saved table: report_ready_multi_indicator_validation_paragraphs.csv
Saved table: methodological_decision_updates_multi_indicator_validation.csv


,topic,report_text
0,Multi-indicator validation 2007,"For the 2007 shock, multi-indicator validation..."
1,Multi-indicator validation 2019,"For the 2019 shock, multi-indicator validation..."
2,Validation design,The multi-indicator validation step evaluates ...
3,Validation variables,"The validation indicators include GDP growth, ..."
4,Interpretation of mismatch cases,Mismatch cases are expected in a multidimensio...


,decision,choice,reason,risk_controlled
0,Use multi-indicator validation after POSet con...,"GDP growth, unemployment, inflation, public de...",Checks whether structural POSet position align...,Avoids relying only on GDP recovery years.
1,Keep validation indicators outside ordering,validation only,They are post-shock outcomes or macro outcome ...,Prevents outcome leakage into the POSet.
2,Use shock-specific post windows,2008–2012 for 2007 shock; 2020–2023 for 2019 s...,Captures near/post-shock macro adjustment wind...,Separates structural baseline from validation ...


In [11]:

# ------------------------------------------------------
# SIMPLE FIGURES
# ------------------------------------------------------

if len(frontier_vs_nonfrontier_multi_indicator_validation):
    for shock_id, group in frontier_vs_nonfrontier_multi_indicator_validation.groupby("shock_id"):
        plot_df = group.copy().sort_values("frontier_advantage_direction_aligned", ascending=True)

        fig, ax = plt.subplots(figsize=(10, 6))
        y = np.arange(len(plot_df))

        ax.barh(y, plot_df["frontier_advantage_direction_aligned"])
        ax.set_yticks(y)
        ax.set_yticklabels(plot_df["metric_label"], fontsize=8)
        ax.axvline(0, linewidth=1)
        ax.set_title(f"Multi-indicator validation frontier advantage — shock {shock_id}")
        ax.set_xlabel("Frontier advantage, direction-aligned")
        ax.grid(True, axis="x", alpha=0.25)

        ax.text(
            0.01,
            -0.14,
            "Positive values mean frontier countries perform better than non-frontier countries on the direction-aligned metric.",
            transform=ax.transAxes,
            fontsize=8,
            ha="left",
        )

        save_figure(
            fig,
            f"multi_indicator_frontier_advantage_shock_{shock_id}.png",
            f"Multi-indicator frontier advantage shock {shock_id}",
            "Frontier vs non-frontier advantage across direction-aligned validation metrics.",
        )

if len(multi_indicator_validation_takeaways):
    fig, ax = plt.subplots(figsize=(8, 5))

    x = np.arange(len(multi_indicator_validation_takeaways))
    ax.bar(x, multi_indicator_validation_takeaways["frontier_better_share"])
    ax.set_xticks(x)
    ax.set_xticklabels(multi_indicator_validation_takeaways["shock_id"].astype(str))
    ax.set_ylim(0, 1.05)
    ax.set_title("Share of validation metrics favouring frontier countries")
    ax.set_xlabel("Shock")
    ax.set_ylabel("Share of available metrics")
    ax.grid(True, axis="y", alpha=0.25)

    save_figure(
        fig,
        "multi_indicator_frontier_better_share_by_shock.png",
        "Multi-indicator frontier better share by shock",
        "Share of available validation metrics where frontier countries outperform non-frontier countries.",
    )


Saved figure: multi_indicator_frontier_advantage_shock_2007.png
Saved figure: multi_indicator_frontier_advantage_shock_2019.png
Saved figure: multi_indicator_frontier_better_share_by_shock.png


In [12]:

# ------------------------------------------------------
# INVENTORIES AND AUDIT WORKBOOK
# ------------------------------------------------------

table_inventory_df = pd.DataFrame(table_inventory)
figure_inventory_df = pd.DataFrame(figure_inventory)

for table, file_name in [
    (table_inventory_df, "multi_indicator_validation_table_inventory.csv"),
    (figure_inventory_df, "multi_indicator_validation_figure_inventory.csv"),
]:
    table.to_csv(PROCESSED_DIR / file_name, index=False)
    table.to_csv(DIAGNOSTICS_DIR / file_name, index=False)
    table.to_csv(TABLES_DIR / file_name, index=False)

run_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "created_at": RUN_TIMESTAMP,
    "countries": multi_indicator_validation_country_level["Country"].nunique(),
    "shock_ids": ",".join(sorted(multi_indicator_validation_country_level["shock_id"].dropna().astype(str).unique())),
    "available_metrics": frontier_vs_nonfrontier_multi_indicator_validation["metric_id"].nunique() if len(frontier_vs_nonfrontier_multi_indicator_validation) else 0,
    "country_metric_rows": len(multi_indicator_validation_country_level),
    "mismatch_cases": len(multi_indicator_validation_mismatch_cases),
    "tables_created": len(table_inventory_df),
    "figures_created": len(figure_inventory_df),
    "note": "Multi-indicator validation uses post-shock outcomes and does not affect POSet ordering.",
}])

run_summary.to_csv(PROCESSED_DIR / "multi_indicator_validation_run_summary.csv", index=False)
run_summary.to_csv(DIAGNOSTICS_DIR / "multi_indicator_validation_run_summary.csv", index=False)
run_summary.to_csv(TABLES_DIR / "multi_indicator_validation_run_summary.csv", index=False)

audit_xlsx = AUDIT_DIR / "16_Multi_Indicator_Shock_Validation_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    resolved_variable_table.to_excel(writer, sheet_name="resolved_variables", index=False)
    multi_indicator_validation_takeaways.to_excel(writer, sheet_name="takeaways", index=False)
    validation_evidence_matrix.to_excel(writer, sheet_name="evidence_matrix", index=False)
    frontier_vs_nonfrontier_multi_indicator_validation.to_excel(writer, sheet_name="frontier_validation", index=False)
    layer_multi_indicator_validation_summary.to_excel(writer, sheet_name="layer_validation", index=False)
    layer_multi_indicator_trend_summary.to_excel(writer, sheet_name="layer_trend", index=False)
    multi_indicator_validation_mismatch_cases.to_excel(writer, sheet_name="mismatch_cases", index=False)
    multi_indicator_validation_synthesis.to_excel(writer, sheet_name="synthesis", index=False)
    report_ready_multi_indicator_validation_paragraphs.to_excel(writer, sheet_name="report_paragraphs", index=False)
    methodological_decision_updates_multi_indicator_validation.to_excel(writer, sheet_name="decision_updates", index=False)
    table_inventory_df.to_excel(writer, sheet_name="table_inventory", index=False)
    figure_inventory_df.to_excel(writer, sheet_name="figure_inventory", index=False)

print("Audit workbook:", audit_xlsx.resolve())
display(run_summary)
display(table_inventory_df)
display(figure_inventory_df)


Audit workbook: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\16_Multi_Indicator_Shock_Validation_Audit.xlsx


,run_id,created_at,countries,shock_ids,available_metrics,country_metric_rows,mismatch_cases,tables_created,figures_created,note
0,20260624_220702,2026-06-24 22:07:02,35,"2007,2019",6,360,63,11,3,Multi-indicator validation uses post-shock out...


,file_name,title,description,rows,columns,processed_path,diagnostics_path,table_path,created_at
0,multi_indicator_resolved_variables.csv,Resolved validation variables,Maps validation variable keys to columns avail...,5,4,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
1,multi_indicator_validation_country_level.csv,Multi-indicator validation country level,Country-metric-level validation dataset for po...,360,23,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
2,frontier_vs_nonfrontier_multi_indicator_valida...,Frontier vs non-frontier multi-indicator valid...,Compares post-shock macro outcomes between POS...,12,14,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
3,layer_multi_indicator_validation_summary.csv,Layer multi-indicator validation summary,Post-shock macro validation outcomes summarize...,54,12,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
4,layer_multi_indicator_trend_summary.csv,Layer multi-indicator trend summary,Spearman-style layer trend for direction-align...,12,6,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
5,multi_indicator_validation_takeaways.csv,Multi-indicator validation takeaways,Shock-level validation takeaways across all av...,2,8,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
6,validation_evidence_matrix.csv,Validation evidence matrix,Metric-level evidence matrix for frontier vs n...,12,7,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
7,multi_indicator_validation_synthesis.csv,Multi-indicator validation synthesis,High-level synthesis of multi-indicator valida...,4,3,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
8,multi_indicator_validation_mismatch_cases.csv,Multi-indicator validation mismatch cases,Country-metric cases where structural POSet po...,63,11,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
9,report_ready_multi_indicator_validation_paragr...,Report-ready multi-indicator validation paragr...,Report-ready explanatory text for multi-indica...,5,2,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02


,file_name,title,description,path,created_at
0,multi_indicator_frontier_advantage_shock_2007.png,Multi-indicator frontier advantage shock 2007,Frontier vs non-frontier advantage across dire...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
1,multi_indicator_frontier_advantage_shock_2019.png,Multi-indicator frontier advantage shock 2019,Frontier vs non-frontier advantage across dire...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02
2,multi_indicator_frontier_better_share_by_shock...,Multi-indicator frontier better share by shock,Share of available validation metrics where fr...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 22:07:02


In [13]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

expected_outputs = [
    "multi_indicator_resolved_variables.csv",
    "multi_indicator_validation_country_level.csv",
    "frontier_vs_nonfrontier_multi_indicator_validation.csv",
    "layer_multi_indicator_validation_summary.csv",
    "layer_multi_indicator_trend_summary.csv",
    "multi_indicator_validation_takeaways.csv",
    "validation_evidence_matrix.csv",
    "multi_indicator_validation_synthesis.csv",
    "multi_indicator_validation_mismatch_cases.csv",
    "report_ready_multi_indicator_validation_paragraphs.csv",
    "methodological_decision_updates_multi_indicator_validation.csv",
    "multi_indicator_validation_run_summary.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("16 MULTI-INDICATOR SHOCK VALIDATION COMPLETE")
print("=" * 90)

display(output_check)

print("\nImportant interpretation:")
print("This is descriptive validation using post-shock macro outcomes. It is not part of POSet ordering.")

print("\nKey outputs to inspect/send:")
print("- 16_Multi_Indicator_Shock_Validation_Audit.xlsx")
print("- multi_indicator_validation_takeaways.csv")
print("- validation_evidence_matrix.csv")
print("- frontier_vs_nonfrontier_multi_indicator_validation.csv")
print("- multi_indicator_validation_mismatch_cases.csv")

print("\nNext notebook:")
print("17_Validation_Synthesis_and_Final_Interpretation_2002_5Var.ipynb")


16 MULTI-INDICATOR SHOCK VALIDATION COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,multi_indicator_resolved_variables.csv,True,True,True
1,multi_indicator_validation_country_level.csv,True,True,True
2,frontier_vs_nonfrontier_multi_indicator_valida...,True,True,True
3,layer_multi_indicator_validation_summary.csv,True,True,True
4,layer_multi_indicator_trend_summary.csv,True,True,True
5,multi_indicator_validation_takeaways.csv,True,True,True
6,validation_evidence_matrix.csv,True,True,True
7,multi_indicator_validation_synthesis.csv,True,True,True
8,multi_indicator_validation_mismatch_cases.csv,True,True,True
9,report_ready_multi_indicator_validation_paragr...,True,True,True



Important interpretation:
This is descriptive validation using post-shock macro outcomes. It is not part of POSet ordering.

Key outputs to inspect/send:
- 16_Multi_Indicator_Shock_Validation_Audit.xlsx
- multi_indicator_validation_takeaways.csv
- validation_evidence_matrix.csv
- frontier_vs_nonfrontier_multi_indicator_validation.csv
- multi_indicator_validation_mismatch_cases.csv

Next notebook:
17_Validation_Synthesis_and_Final_Interpretation_2002_5Var.ipynb
